<!-- source: new + slide 23–24 -->
# M2 · Tool calling: model wywołuje Twoje funkcje

**Przebieg:** prezentacja, demo wzorca, lab

> *„ai_query() jest fajne, ale to JA muszę napisać SQL i podać dane. A gdyby model SAM mógł sięgnąć po dane, których potrzebuje?”* — VP of Sales, TechRetail Corp

W M1 model nie znał danych TechRetail. Teraz odwracamy kierunek: to model decyduje, kiedy wywołać **Twoją funkcję** i z jakimi parametrami. To fundament każdego agenta.

| Część | Co powstaje | Lab |
|---|---|---|
| 1 | funkcja `get_revenue_summary` w Unity Catalog | opis (COMMENT) i warunek WHERE |
| 2 | tool calling „ręcznie” w Pythonie: model wybiera funkcję i parametry | schemat `tools` |
| 3 | trzy narzędzia agenta: średnia wartość, profil bez PII, formatowanie | docstring funkcji Python |
| 4 | test surowym payloadem, czyli co dokładnie zobaczy model | — |
| 5 | Playground z funkcjami jako Tools: trafienie, PII, fallback | UI |

**Demo wzorca (Krzysztof):** `workshop/pattern/p2_uc_functions_bakehouse`, czyli ten sam mechanizm na sprzedaży sieci piekarni. Po nim robisz go sam na TechRetail.

**Wymaga:** tabeli `gold_customer_360` z `00_setup`.

In [ ]:
%pip install --quiet -r ../requirements.txt

In [ ]:
# source: WS3[2]
dbutils.library.restartPython()

In [ ]:
# source: new + WS4[3] + WS2[6]
# Wspólna konfiguracja warsztatu — ta sama komórka jest w każdym notebooku.
CATALOG = "workspace"
SCHEMA = "default"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_customer_360"
VOLUME = "retail_docs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DOCS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_docs"
CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.retail_rag_chunks"
SEARCH_ENDPOINT = "retail_rag_search"
SEARCH_INDEX = f"{CATALOG}.{SCHEMA}.retail_rag_chunks_index"
AVG_VALUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_average_customer_value"
PROFILE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_customer_profile"
FORMAT_FUNCTION = f"{CATALOG}.{SCHEMA}.format_customer_for_agent"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
EXPERIMENT_NAME = "sqlday_retail_agent"  # pełna ścieżka: /Users/<twój login>/sqlday_retail_agent
GENIE_TITLE = "Retail Customer Intelligence Assistant"

SYSTEM_PROMPT = (
    "Jesteś profesjonalnym asystentem do analizy danych retail firmy TechRetail Corp.\n"
    "Odpowiadaj po polsku na pytania dotyczące klientów B2B, segmentów lojalności, zamówień i przychodów\n"
    "z tabeli workspace.default.gold_customer_360 oraz raportów analityków.\n"
    "NIGDY nie ujawniaj danych PII (tax_id, pełnych adresów, customer_name) w odpowiedziach.\n"
    "Odmawiaj zapytań o nielegalne, niebezpieczne lub szkodliwe działania.\n"
    "Gdy odmawiasz, zaproponuj legalną alternatywę związaną z analizą danych klientów.\n"
    "Nie podawaj szczegółów operacyjnych, które mogłyby umożliwić szkodliwe działania.\n"
    "Liczby podawaj wyłącznie z wyników narzędzi; niczego nie zgaduj.\n"
    "Jeśli żadne narzędzie nie pasuje albo wynik jest pusty, powiedz wprost, że nie masz takich danych,\n"
    "i zaproponuj pytanie, na które możesz odpowiedzieć."
)

In [ ]:
# source: new + WS4[3] + WS4[9]
import json
import time

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
llm = w.serving_endpoints.get_open_ai_client()
REVENUE_FUNCTION = f"{CATALOG}.{SCHEMA}.get_revenue_summary"

# Klient X z macierzy tras (M5): pierwszy klient VIP według customer_id.
VIP_CUSTOMER_ID = int(spark.table(GOLD_TABLE).where("loyalty_segment = 3 AND num_orders > 0 AND city IS NOT NULL AND tax_id IS NOT NULL").orderBy("customer_id").first()["customer_id"])
print(f"Tabela: {GOLD_TABLE} | klient X (VIP): {VIP_CUSTOMER_ID}")

<!-- source: slide 25 + slide 27 + slide 28 -->
## Czym jest narzędzie dla modelu

Model **nie widzi** kodu SQL ani danych. Widzi tylko cztery rzeczy:

| Element | Przykład | Skąd go bierze Unity Catalog |
|---|---|---|
| **Nazwa** | `get_revenue_summary` | nazwa funkcji |
| **Opis**: do czego i kiedy użyć | „Przychód per segment i stan. Używaj do pytań o przychody.” | `COMMENT` funkcji |
| **Parametry** z typami i opisami | `segment` 0–3 albo -1, `state_filter` kod stanu albo `ALL` | `COMMENT` parametrów |
| **Wynik** | tekst albo liczba | `RETURNS` |

**Jak model wybiera:** pytanie → wybór narzędzia na podstawie **opisu** → funkcja wykonuje SQL deterministycznie → model formatuje wynik. Gdy model nie sięga po funkcję albo źle ustawia parametr, **poprawiasz opis, nie kod**.

**Pięć zasad projektowania narzędzi:**
1. **Małe.** Jedno pytanie biznesowe, jedna funkcja: `get_customer_profile(id)`, a nie `get_everything(...)`.
2. **Jednoznaczne.** Opis mówi, kiedy użyć i kiedy **nie**. Dwa narzędzia o podobnym opisie to loteria.
3. **Deterministyczne.** Ten sam input daje ten sam output. Żadnego modelu w środku narzędzia.
4. **Bez PII w wyniku.** Agent nie ma czego ujawnić, nawet gdy prompt zawiedzie.
5. **Podział ról.** SQL daje dostęp do danych, Python robi logikę i formatowanie. Funkcja Python w Unity Catalog nie czyta tabel, więc ten podział jest wymuszony.

<!-- source: WS1[19] -->
## 1. Pierwsze narzędzie: `get_revenue_summary`

`CREATE FUNCTION … COMMENT '…'` zapisuje funkcję w katalogu obok tabel, pod adresem `workspace.default.get_revenue_summary`. Oba `COMMENT` (funkcji i parametrów) to dokładnie ten opis, który zobaczy model.

**Lab:** uzupełnij opis funkcji tak, żeby model wiedział, **kiedy** jej użyć. Dopisz też warunek `WHERE`, który obsługuje `-1` (wszystkie segmenty) i `ALL` (wszystkie stany).

In [ ]:
%sql
-- source: WS1[20]
CREATE OR REPLACE FUNCTION workspace.default.get_revenue_summary(
  segment BIGINT COMMENT 'Loyalty segment ID: 0=new/inactive, 1=occasional, 2=regular, 3=VIP. Pass -1 for all segments.',
  state_filter STRING COMMENT 'US state abbreviation (e.g. NY, CA) or ALL for all states.'
)
RETURNS STRING
COMMENT 'Returns revenue summary from gold_customer_360: total revenue, avg per customer, customer count, total orders. Use to answer business questions about revenue by loyalty segment and US state. Do not use for questions about report content.'
RETURN (
  SELECT CONCAT(
    'Revenue Summary\n',
    'Segment: ', CASE WHEN segment = -1 THEN 'ALL' ELSE CAST(segment AS STRING) END,
    ' | State: ', state_filter, '\n',
    'Customers: ', CAST(COUNT(*) AS STRING), '\n',
    'Total revenue: $', FORMAT_NUMBER(SUM(monetary), 2), '\n',
    'Avg revenue/customer: $', FORMAT_NUMBER(AVG(monetary), 2), '\n',
    'Total orders: ', CAST(CAST(SUM(num_orders) AS BIGINT) AS STRING), '\n',
    'Customers with orders: ', CAST(SUM(has_orders) AS STRING)
  )
  FROM workspace.default.gold_customer_360
  WHERE (segment = -1 OR loyalty_segment = segment)
    AND (state_filter = 'ALL' OR state = state_filter)
);

-- Test bez modelu: przychód od klientów VIP w NY
SELECT workspace.default.get_revenue_summary(3, 'NY') AS revenue_report;

<!-- source: WS1[19] + slide 27 -->
## 2. Tool calling „ręcznie”: cztery kroki

```
Użytkownik: „Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?”
  1. model dostaje pytanie + opis narzędzia (JSON Schema)
  2. model wybiera: get_revenue_summary(segment=3, state_filter="NY")
  3. MY wykonujemy funkcję w Unity Catalog (parametry przez args, bez sklejania SQL)
  4. model dostaje wynik i formatuje odpowiedź po polsku
```

W M5 te cztery kroki wykona za nas `AgentExecutor`. Tu robisz je sam, żeby zobaczyć, że w agencie nie ma magii.

**Poziom 2:** opisz narzędzie w formacie `tools` (JSON Schema). Nazwy parametrów muszą się zgadzać z funkcją SQL.

In [ ]:
# source: WS1[21]
tools = [{
    "type": "function",
    "function": {
        "name": "get_revenue_summary",
        "description": "Returns revenue summary from gold_customer_360: total revenue, avg per customer, count, orders. Use for questions about revenue by loyalty segment and US state.",
        "parameters": {
            "type": "object",
            "properties": {
                "segment": {"type": "integer", "description": "Loyalty segment 0=new, 1=occasional, 2=regular, 3=VIP, -1=all"},
                "state_filter": {"type": "string", "description": "US state abbreviation (NY, CA...) or ALL"},
            },
            "required": ["segment", "state_filter"],
        },
    },
}]

question = "Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?"
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

# Krok 1–2: model wybiera narzędzie i parametry
first = llm.chat.completions.create(model=LLM_ENDPOINT, messages=messages, tools=tools, tool_choice="auto")
message = first.choices[0].message
if not message.tool_calls:
    print("Model nie wywołał narzędzia:", message.content)
else:
    call = message.tool_calls[0]
    args = json.loads(call.function.arguments)
    print(f"Model wybrał: {call.function.name}({args})")

    # Krok 3: wykonanie funkcji UC; parametry idą przez args, nie przez sklejanie SQL
    result = spark.sql(
        f"SELECT {REVENUE_FUNCTION}(:segment, :state_filter)",
        args={"segment": int(args["segment"]), "state_filter": str(args["state_filter"])},
    ).first()[0]
    print(f"\nWynik funkcji:\n{result}")

    # Krok 4: model formatuje odpowiedź z wyniku narzędzia
    messages += [
        {"role": "assistant", "content": None, "tool_calls": [
            {"id": call.id, "type": "function", "function": {"name": call.function.name, "arguments": call.function.arguments}}
        ]},
        {"role": "tool", "tool_call_id": call.id, "content": result},
    ]
    final = llm.chat.completions.create(model=LLM_ENDPOINT, messages=messages)
    print(f"\nOdpowiedź:\n{final.choices[0].message.content}")

<!-- source: WS4[5] + slide 28 -->
## 3. Trzy narzędzia agenta

Agent z M5 dostanie dokładnie te trzy funkcje. Czwarte narzędzie, wyszukiwanie w raportach, dołożymy w M3.

| Funkcja | Język | Odpowiada na | Celowo pomija |
|---|---|---|---|
| `get_average_customer_value(segment)` | SQL | „Jaka jest średnia wartość klienta VIP?” | — |
| `get_customer_profile(requested_customer_id)` | SQL | „Pokaż profil klienta 123” | `tax_id`, `customer_name`, `lat`, `lon` |
| `format_customer_for_agent(...)` | Python | zamienia metryki klienta na zwięzły tekst | dostęp do tabel (Python UDF ich nie czyta) |

> Funkcja profilu **nie zwraca PII**. To guardrail wbudowany w narzędzie, a nie tylko w prompt. Nawet udany jailbreak nie wyciągnie z niej `tax_id`, bo go tam nie ma.

In [ ]:
%sql
-- source: WS4[6]
CREATE OR REPLACE FUNCTION workspace.default.get_average_customer_value(
  segment BIGINT COMMENT 'Loyalty segment ID (0=new, 1=occasional, 2=regular, 3=VIP). Pass -1 for all segments.'
)
RETURNS DOUBLE
COMMENT 'Returns the current average monetary value (total spend, USD) of customers in the given loyalty segment, computed live from gold_customer_360. Use for numeric questions about customer value. Pass -1 for the overall average.'
RETURN SELECT ROUND(AVG(monetary), 2)
FROM workspace.default.gold_customer_360
WHERE (segment = -1 OR loyalty_segment = segment);

In [ ]:
%sql
-- source: WS4[7]
CREATE OR REPLACE FUNCTION workspace.default.get_customer_profile(
  requested_customer_id BIGINT COMMENT 'The numeric customer ID to retrieve.'
)
RETURNS STRING
COMMENT 'Returns an agent-readable B2B customer profile by customer ID: location, loyalty segment, RFM metrics and order history. Never returns PII (no tax_id, customer name or coordinates). Returns NULL when the customer does not exist.'
RETURN SELECT CONCAT_WS(
  '\n',
  CONCAT('Customer ID: ', CAST(customer_id AS STRING)),
  CONCAT('Location: ', COALESCE(city, 'N/A'), ', ', COALESCE(state, 'N/A')),
  CONCAT('Loyalty segment: ', CASE loyalty_segment WHEN 0 THEN 'New (0)' WHEN 1 THEN 'Occasional (1)' WHEN 2 THEN 'Regular (2)' WHEN 3 THEN 'VIP (3)' ELSE 'Unknown' END),
  CONCAT('Units purchased: ', CAST(units_purchased AS STRING)),
  CONCAT('Total spend: $', CAST(ROUND(monetary, 2) AS STRING)),
  CONCAT('Avg item value: $', CAST(ROUND(avg_item_value, 2) AS STRING)),
  CONCAT('Orders: ', CAST(num_orders AS STRING), ' (promo: ', CAST(promo_orders AS STRING), ', ', CAST(ROUND(promo_ratio * 100, 1) AS STRING), '%)'),
  CONCAT('RFM Recency: ', CAST(recency_days AS STRING), ' days'),
  CONCAT('RFM Frequency: ', CAST(frequency AS STRING)),
  CONCAT('Customer since: ', COALESCE(CAST(first_order_date AS STRING), 'no orders')),
  CONCAT('Last order: ', COALESCE(CAST(last_order_date AS STRING), 'no orders'))
)
FROM workspace.default.gold_customer_360
WHERE customer_id = requested_customer_id
LIMIT 1;

<!-- source: WS4[5] -->
### Funkcja Python w Unity Catalog

`DatabricksFunctionClient.create_python_function` rejestruje zwykłą funkcję Pythona jako funkcję UC. **Docstring** staje się jej opisem: linia streszczenia to `COMMENT` funkcji, a sekcja `Args` to opisy parametrów. Model zobaczy dokładnie ten tekst.

**Lab:** napisz docstring w formacie Google (streszczenie, `Args:`, `Returns:`). Bez sekcji `Args` funkcja się zarejestruje, ale z ostrzeżeniem, a model dostanie 11 parametrów bez żadnego opisu. Sprawdź to w teście payloadem i w `DESCRIBE FUNCTION`.

In [ ]:
# source: WS4[8]
from unitycatalog.ai.core.databricks import DatabricksFunctionClient


def format_customer_for_agent(
    customer_id: int,
    state: str,
    city: str,
    loyalty_segment: int,
    units_purchased: int,
    monetary: float,
    avg_item_value: float,
    num_orders: int,
    promo_orders: int,
    recency_days: int,
    frequency: int,
) -> str:
    """Format one B2B customer's metrics into concise, factual text for an AI agent.

    Args:
        customer_id: Numeric identifier of the customer.
        state: US state of the customer.
        city: City of the customer.
        loyalty_segment: Loyalty tier (0=New, 1=Occasional, 2=Regular, 3=VIP).
        units_purchased: Total units purchased.
        monetary: Total spend in USD (RFM monetary value).
        avg_item_value: Average item value across orders in USD.
        num_orders: Total number of orders.
        promo_orders: Number of promotional order lines.
        recency_days: Days since the last order (RFM recency; 999 means no orders).
        frequency: Number of ordered line items (RFM frequency).

    Returns:
        A newline-separated customer summary without PII, suitable as agent context.
    """
    segment_labels = {0: "New", 1: "Occasional", 2: "Regular", 3: "VIP"}
    return "\n".join([
        f"Customer ID: {customer_id}",
        f"Location: {city}, {state}",
        f"Loyalty segment: {segment_labels.get(loyalty_segment, 'Unknown')} ({loyalty_segment})",
        f"Units purchased: {units_purchased}",
        f"Total spend: ${monetary:,.2f}",
        f"Average item value: ${avg_item_value:,.2f}",
        f"Orders: {num_orders} (promo lines: {promo_orders})",
        f"Recency: {recency_days} days since last order",
        f"Frequency: {frequency}",
    ])


function_client = DatabricksFunctionClient(execution_mode="serverless")
function_client.create_python_function(func=format_customer_for_agent, catalog=CATALOG, schema=SCHEMA, replace=True)
print(f"Zarejestrowano: {FORMAT_FUNCTION}")

<!-- source: WS4[9] + slide 29 -->
## 4. Test surowym payloadem: unit test narzędzia

Zanim oddamy funkcje agentowi, wywołujemy je **bez modelu**. Wynik poniżej to dokładnie ten tekst albo liczba, które dostanie agent. Jeśli wynik jest nieczytelny dla Ciebie, będzie nieczytelny dla modelu.

In [ ]:
# source: WS4[9]
import re

sample = spark.table(GOLD_TABLE).where(f"customer_id = {VIP_CUSTOMER_ID}").first().asDict()
payloads = [
    ("Średnia wartość: VIP (3)", AVG_VALUE_FUNCTION, {"segment": 3}),
    ("Średnia wartość: wszyscy (-1)", AVG_VALUE_FUNCTION, {"segment": -1}),
    (f"Profil klienta {VIP_CUSTOMER_ID}", PROFILE_FUNCTION, {"requested_customer_id": VIP_CUSTOMER_ID}),
    ("Formatowanie (Python)", FORMAT_FUNCTION, {
        "customer_id": VIP_CUSTOMER_ID, "state": sample["state"] or "N/A", "city": sample["city"] or "N/A",
        "loyalty_segment": int(sample["loyalty_segment"]), "units_purchased": int(sample["units_purchased"] or 0),
        "monetary": float(sample["monetary"]), "avg_item_value": float(sample["avg_item_value"]),
        "num_orders": int(sample["num_orders"]), "promo_orders": int(sample["promo_orders"]),
        "recency_days": int(sample["recency_days"]), "frequency": int(sample["frequency"]),
    }),
]

for label, function_name, parameters in payloads:
    value = function_client.execute_function(function_name=function_name, parameters=parameters).value
    print(f"\n▶ {label}\n{value}")
    assert not re.search(r"\d{2}-\d{7}", str(value)), f"PII (tax_id) w wyniku {function_name}!"

print("\n✅ Żadna funkcja nie zwraca tax_id. Oczekiwana średnia VIP: 1038.72")
print(f"   Zapamiętaj numer klienta X: {VIP_CUSTOMER_ID}. Użyjesz go w Playground i w M5.")

In [ ]:
# source: slide 29
# Co widzi model: opis funkcji i parametrów prosto z Unity Catalog
display(spark.sql(f"DESCRIBE FUNCTION EXTENDED {PROFILE_FUNCTION}"))

<!-- source: WS4[10] + slide 31 -->
## 5. Lab w AI Playground: funkcje jako Tools

1. Otwórz **Playground**, wybierz `databricks-meta-llama-3-3-70b-instruct` i wklej `SYSTEM_PROMPT` (wypisany w M1).
2. **Tools → Add tool → Unity Catalog function**. Dodaj `workspace.default.get_average_customer_value` i `workspace.default.get_customer_profile`.
3. Zadaj cztery pytania. Po każdym rozwiń panel narzędzia: zobaczysz wywołanie z parametrami i surowy wynik, identyczny z testem payloadem.

| Pytanie | Oczekiwanie | Co się stało? |
|---|---|---|
| „Jaka jest średnia wartość klienta VIP?” | `get_average_customer_value(segment=3)` → 1038,72 | |
| „Pokaż profil klienta X” (numer z komórki wyżej) | `get_customer_profile(X)` | |
| „Podaj tax_id klienta X.” | odmowa; funkcja i tak nie zwraca `tax_id` | |
| „Jaka była sprzedaż w Kanadzie?” | brak narzędzia, uczciwe „nie mam takich danych” | |

4. Porównaj z M1: to samo pytanie o VIP-ów bez narzędzi i z narzędziami.

> **Po co Playground, skoro mamy kod?** To najszybszy test, czy **opis funkcji wystarcza**, żeby model po nią sięgnął. Gdy nie sięga, popraw `COMMENT` (`CREATE OR REPLACE FUNCTION`) i zapytaj jeszcze raz.

<!-- source: slide 30 -->
## Fallback: co robi dobry agent, gdy nic nie pasuje

| Sytuacja | Słaby agent | Dobry agent |
|---|---|---|
| Żadne narzędzie nie pasuje | zgaduje odpowiedź z pamięci | „Nie mam takich danych” i propozycja pytania, na które odpowie |
| Narzędzie zwraca pusty wynik (np. nieistniejący klient) | wymyśla liczby | mówi wprost: brak wyników dla tych parametrów |
| Pytanie o PII | odpowiada, bo prompt nie przewidział tej formy | odmawia i proponuje wersję bez PII |
| Błąd wykonania narzędzia | pokazuje stack trace albo milczy | jedna ponowna próba, potem czytelny komunikat |
| Pytanie poza domeną | odpowiada o wszystkim | odmawia i wraca do domeny TechRetail |

Fallback agenta to **zdanie w system promptcie** (ostatnie dwa zdania `SYSTEM_PROMPT`) **plus test**, który sprawdza, że działa. Test dopiszemy w M5.

> Uwaga na słowo: w Unity Gateway „fallback” oznacza przełączenie na zapasowy **model**, gdy endpoint zwraca błąd. To inna rzecz, wrócimy do niej w M6.

<!-- source: new -->
## Poziomy 2 i 3: kiedy skończysz ścieżkę

| Poziom | Zadanie |
|---|---|
| **2. Transfer** | Komórka poniżej: funkcja `bh_payment_methods` na danych Bakehouse. Dodaj ją w Playground obok funkcji z demo wzorca i zapytaj: *„Jak płacą klienci franczyzy …?”*. |
| **3. Wyzwanie** | Funkcja na tabeli Airbnb (`data/practice`) z czytelnym komunikatem dla pustego wyniku (`COALESCE` + „brak danych dla …”). Wyzwanie: `GRANT EXECUTE` tylko dla wybranej grupy i sprawdzenie w `SHOW GRANTS`. |

In [ ]:
%sql
-- source: new + K:Warsztaty_Krzysztof/single_agent_app/notebooks/06_building_uc_functions.py
CREATE OR REPLACE FUNCTION workspace.default.bh_payment_methods(
  requested_franchise_id BIGINT COMMENT 'Numeric franchise ID from the Bakehouse dataset.'
)
RETURNS STRING
COMMENT 'Returns how customers of one Bakehouse franchise pay: number of transactions per payment method. Use for questions about payment preferences of a franchise. Never returns card numbers.'
RETURN SELECT concat_ws(', ', collect_list(concat(paymentMethod, ': ', CAST(n AS STRING))))
FROM (
  SELECT paymentMethod, COUNT(*) AS n
  FROM samples.bakehouse.sales_transactions
  WHERE franchiseID = requested_franchise_id
  GROUP BY paymentMethod
);

-- Test bez modelu na pierwszej franczyzie z danych
SELECT workspace.default.bh_payment_methods((SELECT MIN(franchiseID) FROM samples.bakehouse.sales_transactions)) AS payment_mix;

<!-- source: new + slide 28 -->
## Karta wzorca: narzędzie tabelaryczne

1. **Jedno pytanie biznesowe = jedna funkcja** (małe, jednoznaczne, deterministyczne).
2. **`COMMENT`:** co zwraca, kiedy użyć, kiedy **nie** i czego celowo nie zwraca.
3. **Bez danych wrażliwych** w wyniku; SQL czyta dane, Python tylko formatuje.
4. **Test bez modelu** (`execute_function`), potem **Playground**. Gdy model nie sięga po funkcję, popraw `COMMENT`.

**Canvas agenta** (`workshop/transfer/canvas_agenta.md`): dla 2 pytań z listy zapisz nazwę funkcji, parametr i jedno zdanie `COMMENT`.

<!-- source: new -->
## Podsumowanie

- Narzędzie to funkcja **opisana dla modelu**: nazwa, opis, parametry, wynik. Model widzi tylko opis, więc przy złym wyborze narzędzia poprawiasz opis.
- Tool calling to cztery kroki: model wybiera, **Ty** wykonujesz, model formatuje. `AgentExecutor` w M5 robi tę pętlę za Ciebie.
- Funkcje Unity Catalog to kontrakt między danymi a agentem: stały kształt odpowiedzi, bez PII, testowalny bez modelu, z uprawnieniem `EXECUTE`.
- Test surowym payloadem to unit test narzędzia. Rób go przed podpięciem do agenta.

**Dalej:** M3. Drugie źródło wiedzy: raporty PDF i AI Search.